"""
This piece of code will be cleaning and transforming the customer data
"""

In [2]:
import pandas as pd


In [3]:


df = pd.read_csv("customer_data.csv")

df.head()


,customer_id,meter_id,customer_type,region,installation_date
0,CUST00001,MTR466611,Residential,Central,2023-12-26
1,CUST00002,MTR764322,Residential,North,2020-07-06
2,CUST00003,MTR900801,Commercial,East,2023-12-27
3,CUST00004,MTR403291,Residential,Central,2016-09-02
4,CUST00005,MTR225586,Residential,South,2022-10-11


CONNECTING TO SNOWFLAKE DATALAKE USING PANDAS

In [17]:
import snowflake.connector
from dotenv import load_dotenv
import os

load_dotenv()

conn = None


try:
    conn = snowflake.connector.connect(
        account = os.getenv("SNOWFLAKE_ACCOUNT"),
        user = os.getenv("SNOWFLAKE_USER"),
        role = os.getenv("SNOWFLAKE_ROLE"),
        warehouse = os.getenv("SNOWFLAKE_WAREHOUSE"),
        database = os.getenv("SNOWFLAKE_DATABASE"),
        schema = os.getenv("SNOWFLAKE_SCHEMA"),
        password = os.getenv("SNOWFLAKE_PASSWORD")
    )
    print("Successfully connected to Snowflake!")
except Exception as e:
    print(f"Error: {e}")



Successfully connected to Snowflake!


TESTING VERSION

In [5]:
cur = conn.cursor() # type: ignore
print(cur.execute("SELECT CURRENT_VERSION()"))
print(cur.fetchone())

cur.execute("select * from customer_data")
print(cur.fetchall())

('10.9.2',)
[]


TESTING THE CONNECTION

In [19]:
try:
    cur = conn.cursor() # type: ignore
    cur.execute("SELECT 1")
    print("Connection OK")
except Exception as e:
    print("Connection failed:", e)

Connection failed: 250002 (08003): Connection is closed


FIXING UPPERCASE ISSUES

In [10]:
df.columns = [c.upper() for c in df.columns]

LOADING TO SNOWFLAKE

In [ ]:
from snowflake.connector.pandas_tools import write_pandas
import traceback


if conn:
    try:
        success, nchunks, nrows, _ = write_pandas(
        conn,
        df,                      # your pandas DataFrame
        table_name="CUSTOMER_DATA",  # target table in Snowflake
        )
        print(success, nrows)
        print(f"Successfully loaded {nrows} rows into Snowflake.")
    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()


True 500
Successfully loaded 500 rows into Snowflake.


CLOSE CONNECTION

In [18]:
try:
    conn.close() # type: ignore
    print("Connection closed.")
except Exception as e:
    print(f"Error closing connection: {e}")

Connection closed.
